In [5]:
!pip install huggingface_hub
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [6]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

In [10]:
from huggingface_hub import login

login("hf_your_token")

In [30]:
!pip install transformers peft accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "Qwen/Qwen2-7B-Instruct"
lora_model = "Daleth-hb/qwen-cuda2hip-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(model, lora_model)

model.eval()

Loading weights: 100%|████████████████████████████████████████████████████████| 339/339 [00:01<00:00, 254.00it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [31]:
prompt = """
Convert the following CUDA code to HIP (ROCm compatible).

CUDA code:

__global__ void add(int *a, int *b, int *c) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    c[idx] = a[idx] + b[idx];
}

Provide the HIP version.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


Convert the following CUDA code to HIP (ROCm compatible).

CUDA code:

__global__ void add(int *a, int *b, int *c) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    c[idx] = a[idx] + b[idx];
}

Provide the HIP version.
To convert the given CUDA code to HIP (ROCm compatible), we need to make several adjustments. The primary differences between CUDA and HIP include function names, memory management, and some syntax changes. Below is the equivalent HIP code:

```cpp
#include <hip/hipKernelLaunch.h>
#include <hip/hipDevice.h>

__global__ void add(int *a, int *b, int *c) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    c[idx] = a[idx] + b[idx];
}

int main() {
    // Allocate and initialize HIP devices
    int *d_a, *d_b, *d_c;
    hipMalloc(&d_a, sizeof(int));
    hipMalloc(&d_b, sizeof(int));
    hipMalloc(&d_c, sizeof(int));

    // Copy data from host to device
    int host_a[10] = { /* Initialize array */ };
    int host_b[10] = { /* Initialize array */ };
    int h

In [32]:
prompt = """
Translate this CUDA program to HIP for ROCm.

#include <cuda_runtime.h>

int main() {
    int *d_a;
    cudaMalloc(&d_a, 1024 * sizeof(int));
    cudaDeviceSynchronize();
    cudaFree(d_a);
}

Return the HIP version.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


Translate this CUDA program to HIP for ROCm.

#include <cuda_runtime.h>

int main() {
    int *d_a;
    cudaMalloc(&d_a, 1024 * sizeof(int));
    cudaDeviceSynchronize();
    cudaFree(d_a);
}

Return the HIP version.
*/

#include <hip/hipKernelApi.h>
#include <hip/hipDevice.h>

int main() {
    int *d_a;
    hipMalloc(&d_a, 1024 * sizeof(int));
    hipDeviceSynchronize();
    hipFree(d_a);
}

In the solution, we've translated the CUDA code snippet to use HIP (Heterogeneous-Compute Interface for Portability) functions provided by ROCm. The key changes include:

1. **Library Inclusion**: Instead of `#include <cuda_runtime.h>`, we include `#include <hip/hipKernelApi.h>` and `#include <hip/hipDevice.h>`. These headers provide access to HIP-specific functions and types.

2. **Memory Allocation**: The `cudaMalloc` function is replaced with `hipMalloc`, which is used for allocating device memory in HIP. Similarly, `cudaFree` is replaced with `hipFree`.

3. **Synchronization**: The `cudaDevic

In [33]:
prompt = """
You are a GPU portability assistant.

Convert the CUDA code to HIP and explain the changes.

CUDA code:

cudaMalloc(&ptr, size);
cudaMemcpy(ptr, host, size, cudaMemcpyHostToDevice);
cudaDeviceSynchronize();

Return:
1) HIP code
2) Explanation
"""

In [34]:

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


You are a GPU portability assistant.

Convert the CUDA code to HIP and explain the changes.

CUDA code:

cudaMalloc(&ptr, size);
cudaMemcpy(ptr, host, size, cudaMemcpyHostToDevice);
cudaDeviceSynchronize();

Return:
1) HIP code
2) Explanation
1) HIP code:

hipMalloc(&ptr, size);
hipMemcpy(ptr, host, size, hipMemcpyHostToDevice);
hipDeviceSynchronize();

Explanation:

The CUDA code has been converted to HIP (Heterogeneous-Compute Interface for Portability) code as follows:

1. `cudaMalloc` function is replaced with `hipMalloc` function. Both functions allocate memory on the device, but `hipMalloc` is part of the HIP API, which provides a unified interface for heterogeneous computing across different architectures like CPUs, GPUs, and other accelerators.

2. `cudaMemcpy` function is replaced with `hipMemcpy` function. This function is used to copy data between the host and device memory. The parameters remain similar: source, destination, number of bytes to copy, and the memory transfer

In [35]:
prompt = """
Explain why CUDA code may not run on ROCm and how HIP solves this problem.

In your answer include:
1. Why CUDA is vendor-specific
2. Why AMD GPUs cannot run CUDA directly
3. How HIP enables portability
4. Example of CUDA vs HIP API mapping
"""

In [36]:

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


Explain why CUDA code may not run on ROCm and how HIP solves this problem.

In your answer include:
1. Why CUDA is vendor-specific
2. Why AMD GPUs cannot run CUDA directly
3. How HIP enables portability
4. Example of CUDA vs HIP API mapping
5. Benefits of using HIP over CUDA

CUDA is proprietary software developed by NVIDIA, which provides a parallel computing platform and programming model for general-purpose computing on NVIDIA GPUs. It is tightly integrated with the NVIDIA GPU architecture, offering optimized performance and features specifically tailored to NVIDIA's hardware. Here are the key points you asked about:

1. **Why CUDA is vendor-specific:**
   CUDA is designed to leverage the capabilities of NVIDIA GPUs, including their specialized instruction sets (like Tensor Cores), memory hierarchy, and co-processing units (like SMs). It includes libraries like cuBLAS, cuFFT, and cuRAND that are optimized for NVIDIA GPUs, taking advantage of their unique features such as shared mem